In [0]:
from pyspark.sql.functions import col

df_silver = spark.table(
    "workspace.civicpulse_silver.complaints_daily"
)

print("Rows:", df_silver.count())
print("Columns:", len(df_silver.columns))

In [0]:
from pyspark.sql.functions import (
    min,
    max,
    datediff,
    count
)

date_range = df_silver.select(
    min("date").alias("min_date"),
    max("date").alias("max_date")
).collect()[0]

min_date = date_range["min_date"]
max_date = date_range["max_date"]

expected_days = (max_date - min_date).days + 1
actual_days = df_silver.select("date").distinct().count()

print("Minimum date:", min_date)
print("Maximum date:", max_date)
print("Expected calendar days:", expected_days)
print("Actual recorded days:", actual_days)
print("Missing calendar days:", expected_days - actual_days)

In [0]:
from pyspark.sql.functions import sequence, explode, lit, to_date

calendar_df = (
    spark.range(1)
    .select(
        explode(
            sequence(
                lit(min_date),
                lit(max_date)
            )
        ).alias("date")
    )
)

missing_dates = (
    calendar_df
    .join(
        df_silver.select("date").distinct(),
        on="date",
        how="left_anti"
    )
    .orderBy("date")
)

print("Missing dates:", missing_dates.count())

display(missing_dates)

In [0]:
from pyspark.sql.functions import (
    sum,
    avg,
    min,
    max,
    stddev,
    percentile_approx
)

volume_stats = df_silver.select(
    sum("complaints_received").alias("total_received"),
    sum("complaints_disposed").alias("total_disposed"),
    avg("complaints_received").alias("avg_daily_received"),
    avg("complaints_disposed").alias("avg_daily_disposed"),
    avg("complaints_pending").alias("avg_daily_pending"),
    min("complaints_pending").alias("min_pending"),
    max("complaints_pending").alias("max_pending"),
    stddev("complaints_received").alias("std_received"),
    stddev("complaints_disposed").alias("std_disposed"),
    percentile_approx(
        "complaints_received", 0.95
    ).alias("p95_received"),
    percentile_approx(
        "complaints_disposed", 0.95
    ).alias("p95_disposed")
)

display(volume_stats)

In [0]:
from pyspark.sql.functions import col

high_demand_days = (
    df_silver
    .filter(col("complaints_received") >= 314)
    .select(
        "date",
        "city_name",
        "complaints_received",
        "complaints_disposed",
        "complaints_pending"
    )
    .orderBy(col("complaints_received").desc())
)

print("Days at or above 95th percentile:", high_demand_days.count())

display(high_demand_days.limit(20))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    sum,
    avg,
    max,
    count
)

rolling_7 = Window.orderBy("date").rowsBetween(-6, 0)
rolling_30 = Window.orderBy("date").rowsBetween(-29, 0)

df_rolling = (
    df_silver
    .withColumn(
        "received_7d",
        sum("complaints_received").over(rolling_7)
    )
    .withColumn(
        "avg_received_7d",
        avg("complaints_received").over(rolling_7)
    )
    .withColumn(
        "received_30d",
        sum("complaints_received").over(rolling_30)
    )
    .withColumn(
        "avg_received_30d",
        avg("complaints_received").over(rolling_30)
    )
    .withColumn(
        "max_pending_7d",
        max("complaints_pending").over(rolling_7)
    )
)

display(
    df_rolling.select(
        "date",
        "complaints_received",
        "complaints_pending",
        "received_7d",
        "avg_received_7d",
        "received_30d",
        "avg_received_30d",
        "max_pending_7d"
    ).orderBy("date").limit(35)
)

In [0]:
from pyspark.sql.functions import (
    when,
    round
)

df_pressure = (
    df_rolling
    .withColumn(
        "demand_surge_ratio",
        when(
            col("avg_received_30d") > 0,
            round(
                col("complaints_received") /
                col("avg_received_30d"),
                2
            )
        ).otherwise(None)
    )
    .withColumn(
        "pending_pressure_ratio",
        when(
            col("avg_received_30d") > 0,
            round(
                col("complaints_pending") /
                col("avg_received_30d"),
                2
            )
        ).otherwise(None)
    )
    .withColumn(
        "high_demand_flag",
        when(
            col("complaints_received") >= 314,
            1
        ).otherwise(0)
    )
)

display(
    df_pressure
    .select(
        "date",
        "complaints_received",
        "complaints_pending",
        "avg_received_30d",
        "demand_surge_ratio",
        "pending_pressure_ratio",
        "high_demand_flag"
    )
    .orderBy(col("demand_surge_ratio").desc())
    .limit(20)
)

In [0]:
from pyspark.sql.functions import sum

high_demand_window = Window.orderBy("date").rowsBetween(-6, 0)

df_persistence = (
    df_pressure
    .withColumn(
        "high_demand_days_7d",
        sum("high_demand_flag").over(high_demand_window)
    )
)

display(
    df_persistence
    .select(
        "date",
        "complaints_received",
        "complaints_pending",
        "high_demand_flag",
        "high_demand_days_7d"
    )
    .filter(col("high_demand_days_7d") > 0)
    .orderBy(col("high_demand_days_7d").desc(), col("date"))
    .limit(20)
)

In [0]:
from pyspark.sql.functions import percentile_approx

pending_p95 = (
    df_silver
    .select(
        percentile_approx(
            "complaints_pending",
            0.95
        ).alias("pending_p95")
    )
    .collect()[0]["pending_p95"]
)

print("95th percentile pending:", pending_p95)

In [0]:
from pyspark.sql.functions import col, when, round

df_pressure_v2 = (
    df_persistence
    .withColumn(
        "demand_pressure_ratio",
        round(
            col("complaints_received") / 314,
            2
        )
    )
    .withColumn(
        "backlog_pressure_ratio",
        round(
            col("complaints_pending") / 8950,
            2
        )
    )
    .withColumn(
        "demand_pressure_flag",
        when(col("complaints_received") >= 314, 1).otherwise(0)
    )
    .withColumn(
        "backlog_pressure_flag",
        when(col("complaints_pending") >= 8950, 1).otherwise(0)
    )
)

display(
    df_pressure_v2
    .select(
        "date",
        "complaints_received",
        "complaints_pending",
        "demand_pressure_ratio",
        "backlog_pressure_ratio",
        "high_demand_days_7d",
        "demand_pressure_flag",
        "backlog_pressure_flag"
    )
    .orderBy(
        col("backlog_pressure_ratio").desc(),
        col("demand_pressure_ratio").desc()
    )
    .limit(20)
)

In [0]:
from pyspark.sql.functions import when

df_classified = (
    df_pressure_v2
    .withColumn(
        "pressure_category",
        when(
            (col("demand_pressure_flag") == 1) &
            (col("backlog_pressure_flag") == 1),
            "COMBINED_PRESSURE"
        )
        .when(
            col("demand_pressure_flag") == 1,
            "DEMAND_ONLY"
        )
        .when(
            col("backlog_pressure_flag") == 1,
            "BACKLOG_ONLY"
        )
        .otherwise(
            "NORMAL_RANGE"
        )
    )
)

display(
    df_classified
    .select(
        "date",
        "complaints_received",
        "complaints_pending",
        "high_demand_days_7d",
        "demand_pressure_flag",
        "backlog_pressure_flag",
        "pressure_category"
    )
    .orderBy(col("date").desc())
    .limit(20)
)

In [0]:
from pyspark.sql.functions import count

category_summary = (
    df_classified
    .groupBy("pressure_category")
    .agg(
        count("*").alias("days")
    )
    .orderBy(col("days").desc())
)

display(category_summary)

In [0]:
from pyspark.sql.functions import round, when

df_gold = (
    df_classified
    .withColumn(
        "disposal_activity_ratio",
        when(
            col("complaints_received") > 0,
            round(
                col("complaints_disposed") /
                col("complaints_received"),
                2
            )
        ).otherwise(None)
    )
)

display(
    df_gold.select(
        "date",
        "complaints_received",
        "complaints_disposed",
        "complaints_pending",
        "disposal_activity_ratio",
        "pressure_category"
    ).orderBy(col("date").desc()).limit(20)
)

In [0]:
gold_table = "workspace.civicpulse_gold.service_intelligence_daily"

(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table)
)

print(f"Gold table created: {gold_table}")

In [0]:
df_gold_table = spark.table(
    "workspace.civicpulse_gold.service_intelligence_daily"
)

print("Gold rows:", df_gold_table.count())
print("Gold columns:", len(df_gold_table.columns))

display(
    df_gold_table
    .orderBy(col("date").desc())
    .limit(10)
)